In [4]:
from Data_Preparation import Dataset
from Data_Preparation.Embedding import embedding_encoder 
from Data_Preparation.Tac import tac
from Modelisation.Baselines.OCSVM import ocsvm
from torch.utils.data import ConcatDataset, DataLoader
import Modelisation.evaluation as ev
from Modelisation.Baselines.CVDD.networks import cvdd_Net, embedding_layer
import Modelisation.Baselines.CVDD.utils as utils
import torch.optim as optim
import numpy as np
import time
import torch
from sklearn.cluster import KMeans
from datasets import concatenate_datasets
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.svm import OneClassSVM
from sklearn.model_selection import KFold

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
"""
Autoencoder for textual anomaly detection on Reuters-21578 (yangwang825 subset)
- Sentence encoding: BERT (mean pooling of last hidden states)
- Autoencoder: simple MLP on top of BERT embeddings
- Training: only on inliers (chosen as most frequent class by default)
- Evaluation: reconstruction error -> ROC AUC
"""

import os
import math
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import numpy as np
import random

In [7]:


# ---------------------------
# Config
# ---------------------------
MODEL_NAME = "bert-base-uncased"
HF_DATASET_ID = "yangwang825/reuters-21578"  # HF dataset used
BATCH_SIZE = 64
EMBED_BATCH = 64  # batch size for BERT embedding step (can be different)
LR = 1e-3
N_EPOCHS = 30
LATENT_DIM = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
THRESHOLD_FROM_PERCENTILE = 95  # optional: threshold set from inlier recon errors percentile

# reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ---------------------------
# Utilities
# ---------------------------
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(SEED)

# Mean pooling helper
def mean_pooling(last_hidden_state, attention_mask):
    # last_hidden_state: (B, L, D), attention_mask: (B, L)
    mask = attention_mask.unsqueeze(-1).float()
    sum_hidden = (last_hidden_state * mask).sum(dim=1)
    lengths = mask.sum(dim=1).clamp(min=1e-9)
    return sum_hidden / lengths

# ---------------------------
# Load dataset and choose inlier
# ---------------------------
print("Loading dataset from Hugging Face:", HF_DATASET_ID)
ds = load_dataset(HF_DATASET_ID)  # has 'train' and 'test' splits

# Check label names (hf dataset feature)
label_feature = ds["train"].features["label"]
if hasattr(label_feature, "names"):
    label_names = label_feature.names
else:
    # fallback: try to infer labels from train examples
    label_names = sorted(list({ex["label"] for ex in ds["train"]}))

print("Label names:", label_names)

# Compute most frequent label in training set and set as inlier by default
train_labels = [ex["label"] for ex in ds["train"]]
label_counts = Counter(train_labels)
most_common_label_id, count = label_counts.most_common(1)[0]
INLIER_LABEL_ID = most_common_label_id
INLIER_LABEL_NAME = label_names[INLIER_LABEL_ID] if isinstance(label_names, (list,tuple)) else str(INLIER_LABEL_ID)
print(f"Automatically chosen inlier class: id={INLIER_LABEL_ID}, name='{INLIER_LABEL_NAME}' (freq={count})")
print("If you want to override, change INLIER_LABEL_ID manually in the script.")

Loading dataset from Hugging Face: yangwang825/reuters-21578


README.md:   0%|          | 0.00/465 [00:00<?, ?B/s]

train.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Label names: ['acq', 'crude', 'earn', 'grain', 'interest', 'money-fx', 'ship', 'trade']
Automatically chosen inlier class: id=2, name='earn' (freq=2840)
If you want to override, change INLIER_LABEL_ID manually in the script.


In [8]:


# ---------------------------
# Prepare texts and labels
# ---------------------------
def get_text(example):
    # dataset preview shows a 'text' field containing the article body; fallback to other keys if not present
    for key in ("text", "body", "article", "content"):
        if key in example:
            return example[key]
    # fallback to joining all string fields
    return " ".join(str(v) for k,v in example.items() if isinstance(v, str))

# Build lists
train_texts = [get_text(ex) for ex in ds["train"]]
train_labels = [ex["label"] for ex in ds["train"]]

test_texts = [get_text(ex) for ex in ds["test"]]
test_labels = [ex["label"] for ex in ds["test"]]

# Convert to binary labels: 0 = inlier, 1 = anomaly
test_binary_labels = [0 if l == INLIER_LABEL_ID else 1 for l in test_labels]

# Filter training set to only inliers (autoencoder trained on normal data)
train_inlier_texts = [t for t,l in zip(train_texts, train_labels) if l == INLIER_LABEL_ID]
print(f"Train inliers count: {len(train_inlier_texts)} / {len(train_texts)} total train samples")

# ---------------------------
# BERT tokenizer & model for embeddings
# ---------------------------
print("Loading tokenizer and BERT model:", MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert.eval()  # we'll use it as feature extractor (no fine-tuning here)

Train inliers count: 2840 / 5485 total train samples
Loading tokenizer and BERT model: bert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [9]:

@torch.no_grad()
def embed_texts(texts, batch_size=EMBED_BATCH):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt", max_length=256)
        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)
        outputs = bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        last_hidden = outputs.last_hidden_state  # (B, L, D)
        pooled = mean_pooling(last_hidden, attention_mask)  # (B, D)
        embeddings.append(pooled.cpu())
    embeddings = torch.cat(embeddings, dim=0)
    return embeddings  # Tensor (N, D)

# Precompute embeddings (train inliers and test set)
print("Computing embeddings for train inliers (this may take some time)...")
train_inlier_embeddings = embed_texts(train_inlier_texts, batch_size=EMBED_BATCH)
print("Computing embeddings for test set...")
test_embeddings = embed_texts(test_texts, batch_size=EMBED_BATCH)

print("Embeddings shapes:", train_inlier_embeddings.shape, test_embeddings.shape)

# ---------------------------
# PyTorch Dataset & Autoencoder model
# ---------------------------
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels=None):
        self.emb = embeddings.float()
        self.labels = labels
    def __len__(self):
        return len(self.emb)
    def __getitem__(self, idx):
        if self.labels is None:
            return self.emb[idx]
        else:
            return self.emb[idx], self.labels[idx]

embedding_dim = train_inlier_embeddings.shape[1]

class AE(nn.Module):
    def __init__(self, input_dim, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        recon = self.decoder(z)
        return recon


Computing embeddings for train inliers (this may take some time)...
Computing embeddings for test set...
Embeddings shapes: torch.Size([2840, 768]) torch.Size([2189, 768])


In [10]:
# ---------------------------
# Train
# ---------------------------
ae = AE(input_dim=embedding_dim, latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(ae.parameters(), lr=LR)
criterion = nn.MSELoss()

train_ds = EmbeddingDataset(train_inlier_embeddings)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

print("Start training autoencoder on inliers only...")
ae.train()
for epoch in range(1, N_EPOCHS+1):
    epoch_loss = 0.0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()
        recon = ae(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch.size(0)
    epoch_loss /= len(train_loader.dataset)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{N_EPOCHS} - Loss: {epoch_loss:.6f}")
print("Training finished.")

# ---------------------------
# Evaluate: reconstruction error -> anomaly score
# ---------------------------
ae.eval()
with torch.no_grad():
    test_emb = test_embeddings.to(DEVICE)
    recon = ae(test_emb)
    # use MSE per-sample as anomaly score
    errors = torch.mean((recon - test_emb) ** 2, dim=1).cpu().numpy()

# If you want a threshold to make a binary detector:
# e.g., set threshold as percentile of training inlier reconstruction errors
with torch.no_grad():
    train_recon = ae(train_inlier_embeddings.to(DEVICE))
    train_errors = torch.mean((train_recon - train_inlier_embeddings.to(DEVICE)) ** 2, dim=1).cpu().numpy()
threshold = np.percentile(train_errors, THRESHOLD_FROM_PERCENTILE)
print(f"Threshold (percentile {THRESHOLD_FROM_PERCENTILE}) on train inliers: {threshold:.6e}")

# ROC AUC (higher is better; uses anomaly label=1)
auc = roc_auc_score(test_binary_labels, errors)
print(f"Test ROC AUC (reconstruction error as score): {auc:.4f}")

# Example: show top anomalous documents (largest reconstruction error)
top_k = 10
idxs = np.argsort(-errors)[:top_k]
print("\nTop anomalous test documents (error, true_label_id, text snippet):")
for i in idxs:
    print(f"error={errors[i]:.6e}, label={test_labels[i]}, snippet={test_texts[i][:200].replace('\\n',' ')}...")

# # Save model
# save_path = "ae_reuters_bert_embeddings.pt"
# torch.save({"model_state_dict": ae.state_dict(), "config": {"embedding_dim": embedding_dim, "latent_dim": LATENT_DIM}}, save_path)
# print("Saved AE to", save_path)

# ---------------------------
# Notes & next steps
# ---------------------------
"""
- By default we used BERT as a frozen feature extractor and trained a light MLP autoencoder.
  If you have GPU and want better features, consider fine-tuning BERT jointly with the AE (requires more memory).
- You can change INLIER_LABEL_ID to force a specific class as normal.
- You can experiment with other pooling strategies (CLS token, weighted pooling, mean of last two layers, SBERT).
- To get better sentence embeddings out-of-the-box, consider using sentence-transformers models (paraphrase-MiniLM etc.).
- The threshold selection strategy can be improved (use validation set, or statistical methods).
"""


Start training autoencoder on inliers only...
Epoch 1/30 - Loss: 0.032220
Epoch 5/30 - Loss: 0.009858
Epoch 10/30 - Loss: 0.007399
Epoch 15/30 - Loss: 0.006080
Epoch 20/30 - Loss: 0.005236
Epoch 25/30 - Loss: 0.004634
Epoch 30/30 - Loss: 0.004214
Training finished.
Threshold (percentile 95) on train inliers: 7.716914e-03
Test ROC AUC (reconstruction error as score): 0.9103

Top anomalous test documents (error, true_label_id, text snippet):
error=5.753420e-02, label=5, snippet=stoltenberg says he assumes monetary cooperation will continue...
error=4.850946e-02, label=5, snippet=reagan says u s allies must honor accords on exchange rate stability...
error=4.512340e-02, label=5, snippet=sumita welcomes u s west german joint confirmation of louvre accord...
error=4.458682e-02, label=6, snippet=pentagon says u s warships begin escorting gulf tanker convoy south from kuwait...
error=4.299616e-02, label=5, snippet=stoltenberg does not rule out central bank intervention to stabilize currencies

'\n- By default we used BERT as a frozen feature extractor and trained a light MLP autoencoder.\n  If you have GPU and want better features, consider fine-tuning BERT jointly with the AE (requires more memory).\n- You can change INLIER_LABEL_ID to force a specific class as normal.\n- You can experiment with other pooling strategies (CLS token, weighted pooling, mean of last two layers, SBERT).\n- To get better sentence embeddings out-of-the-box, consider using sentence-transformers models (paraphrase-MiniLM etc.).\n- The threshold selection strategy can be improved (use validation set, or statistical methods).\n'

In [14]:
import numpy as np

def remove_important_words(texts, tokenizer, bert, percentile=65, max_length=256):
    """
    Supprime les tokens dont le poids d'attention est dans le top 'percentile'%.
    Retourne une liste de nouveaux textes.
    """
    bert.eval()
    modified_texts = []

    for text in tqdm(texts, desc="Removing important words"):
        # Tokenize
        enc = tokenizer(text, return_tensors="pt", truncation=True, padding=False, max_length=max_length)
        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        with torch.no_grad():
            outputs = bert(input_ids=input_ids, attention_mask=attention_mask, output_attentions=True)
            # attentions: list of [num_layers] tensors of shape (batch, num_heads, seq_len, seq_len)
            attn = outputs.attentions[-1]  # dernier bloc
            attn = attn.mean(dim=1).squeeze(0)  # moyenne sur les têtes -> (seq_len, seq_len)

        # importance = somme des poids d'attention que chaque token reçoit (colonne)
        token_importance = attn.sum(dim=0).cpu().numpy()

        # On ignore les tokens [CLS] et [SEP] (premier et dernier)
        if len(token_importance) > 2:
            token_importance = token_importance[1:-1]
            tokens = tokenizer.convert_ids_to_tokens(input_ids[0])[1:-1]
        else:
            tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

        # Seuil pour supprimer les plus importants
        threshold = np.percentile(token_importance, percentile)
        kept_tokens = [t for t, imp in zip(tokens, token_importance) if imp < threshold]

        # Reconstruire le texte propre
        new_text = tokenizer.convert_tokens_to_string(kept_tokens)
        modified_texts.append(new_text)

    return modified_texts


In [15]:
# Supprimer les mots importants (20% les plus attentifs)
print("\nGenerating modified texts (removing top 20% most important words)...")
train_inlier_texts_modified = remove_important_words(train_inlier_texts, tokenizer, bert, percentile=65)
test_texts_modified = remove_important_words(test_texts, tokenizer, bert, percentile=65)



Generating modified texts (removing top 20% most important words)...


Removing important words:   0%|          | 0/2840 [00:00<?, ?it/s]

Removing important words:   0%|          | 0/2189 [00:00<?, ?it/s]

In [16]:
# Embeddings BERT pour ces textes modifiés
print("Computing embeddings for modified train and test sets...")
train_inlier_embeddings_mod = embed_texts(train_inlier_texts_modified, batch_size=EMBED_BATCH)
test_embeddings_mod = embed_texts(test_texts_modified, batch_size=EMBED_BATCH)

# Réentraînement du même AE (architecture identique)
ae_mod = AE(input_dim=train_inlier_embeddings_mod.shape[1], latent_dim=LATENT_DIM).to(DEVICE)
optimizer_mod = torch.optim.Adam(ae_mod.parameters(), lr=LR)

train_loader_mod = DataLoader(EmbeddingDataset(train_inlier_embeddings_mod), batch_size=BATCH_SIZE, shuffle=True)

print("\nTraining AE on modified texts...")
for epoch in range(1, N_EPOCHS+1):
    total_loss = 0
    for batch in train_loader_mod:
        batch = batch.to(DEVICE)
        optimizer_mod.zero_grad()
        recon = ae_mod(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer_mod.step()
        total_loss += loss.item() * batch.size(0)
    total_loss /= len(train_loader_mod.dataset)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{N_EPOCHS} - Loss: {total_loss:.6f}")

# Évaluation
ae_mod.eval()
with torch.no_grad():
    test_emb_mod = test_embeddings_mod.to(DEVICE)
    recon_mod = ae_mod(test_emb_mod)
    errors_mod = torch.mean((recon_mod - test_emb_mod) ** 2, dim=1).cpu().numpy()

auc_mod = roc_auc_score(test_binary_labels, errors_mod)
print(f"\nTest ROC AUC after removing important words: {auc_mod:.4f}")

print(f"\nΔAUC = {auc_mod - auc:.4f} (difference from original)")


Computing embeddings for modified train and test sets...

Training AE on modified texts...
Epoch 1/30 - Loss: 0.040028
Epoch 5/30 - Loss: 0.014035
Epoch 10/30 - Loss: 0.010543
Epoch 15/30 - Loss: 0.008615
Epoch 20/30 - Loss: 0.007638
Epoch 25/30 - Loss: 0.006893
Epoch 30/30 - Loss: 0.006333

Test ROC AUC after removing important words: 0.8419

ΔAUC = -0.0684 (difference from original)


## Test

In [3]:
dataset_name = '20newsgroups'
full_dataset_ = False
preprocessing = True

dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
trainset, testset = dataset.get_splits()


Repo card metadata block was not found. Setting CardData to empty.


In [4]:
type_tac = "ruff"
anomaly_rate = 0.1
inlier_topic = "computer"

inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

data_test = tac.textual_anomaly_contamination(
                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)

# corpus = inlier_dataset_train['text']
# vocab = utils.build_vocab(corpus,min_freq=1)
# tokenizer = None
print(inlier_dataset_train)
print(anomaly_dataset_train)
print(data_test)

Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 2936
})
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 326
})
Dataset({
    features: ['text', 'label', 'label_text', 'topic_label_text', 'anomaly_class'],
    num_rows: 2172
})


In [5]:
attention_size = 150
n_attention_heads = 10
type_emb = 'glove'
lr = 0.01
lr_milestones = (20,30)
n_epochs = 40
lambda_p = 1.0
alpha_scheduler = "logarithmic"

In [6]:
# np.unique(data_test['topic_label_text'], return_counts=True)


model, dl_train, dl_test = utils.cvdd_model_pipeline(inlier_dataset_train, data_test, attention_size, n_attention_heads, 
                                                type_emb, 500, 64, True, tokenizer, vocab)

cvdd_trainer = cvdd_Net.CVDDTrainer(optimizer_name='adam', learning_rate=lr, lr_milestones=(lr_milestones[0], lr_milestones[1]),
                                    n_epochs=n_epochs, lambda_p=lambda_p,
                                    alpha_scheduler=alpha_scheduler, weight_decay=1e-4)

model_trained = cvdd_trainer.train(model, dl_train)
auc, ap, fpr95, _ = cvdd_trainer.test(model_trained, dl_test, ad_score='context_dist_mean')

NameError: name 'tokenizer' is not defined

### ################
voir est ce que le testset doit etre en 90% inlier et 10% anomalie

## Import

In [ ]:
import os
import re

def save_results(dataset_name, inlier_topic, type_emb, auc, ap, fpr95,
                 output_dir="/home/youcefk251/My Thesis/Textual-Anomaly-Detection-Framework/Results",
                 filename="results.txt", overwrite=False):

    os.makedirs(output_dir, exist_ok=True)
    filepath = os.path.join(output_dir, filename)

    existing_content = ""
    if os.path.exists(filepath):
        with open(filepath, "r") as f:
            existing_content = f.read()

    pattern = (
        rf"Dataset:\s*{re.escape(dataset_name)}\s*"
        rf"Inlier class:\s*{re.escape(inlier_topic)}\s*"
        rf"Embedding type:\s*{re.escape(type_emb)}"
    )

    new_block = (
        "========================================\n"
        f"Dataset:        {dataset_name}\n"
        f"Inlier class:   {inlier_topic}\n"
        f"Embedding type: {type_emb}\n"
        "----------------------------------------\n"
        f"AUC:            {auc:.4f}\n"
        f"Avg Precision:  {ap:.4f}\n"
        f"FPR@95:         {fpr95:.4f}\n"
        "========================================\n\n"
    )

    if re.search(pattern, existing_content):
        if overwrite:
            existing_content = re.sub(
                r"========================================\n"
                + pattern
                + r".*?========================================\n\n",
                new_block,
                existing_content,
                flags=re.DOTALL
            )
            print(f"Résultats mis à jour pour ({dataset_name}, {inlier_topic}, {type_emb}).")
        else:
            print(f"Résultats déjà présents, non modifiés : ({dataset_name}, {inlier_topic}, {type_emb}).")
            return
    else:
        existing_content += new_block
        print(f"Nouveaux résultats ajoutés pour ({dataset_name}, {inlier_topic}, {type_emb}).")

    with open(filepath, "w") as f:
        f.write(existing_content)


## Test

In [ ]:
def ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid, n_splits=3, verbose=True):
    best_auc = -np.inf
    best_params = None

    for kernel in param_grid["kernel"]:
        for nu in param_grid["nu"]:
            for gamma in param_grid["gamma"]:
                ocsvm_kwargs = {
                    "nu": nu,
                    "kernel": kernel,
                    "gamma": gamma
                    }
                clf, _, _ = ocsvm.One_Class_SVM(X_train, ocsvm_kwargs)
                # clf.fit(X_train)

                scores_test = clf.decision_function(X_test)
                auc = roc_auc_score(y_true_test, scores_test)

                # print(f"{kernel}, nu={nu}, gamma={gamma} -> AUC={auc:.4f}")

                if auc > best_auc:
                    best_auc = auc
                    best_params = {"kernel": kernel, "nu": nu, "gamma": gamma}

    print("\n✅ Best params:", best_params, " | AUC =", best_auc)
    return best_params, best_auc

In [ ]:
dataset_name = 'reuters'
full_dataset_ = False
preprocessing = True

dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
trainset, testset = dataset.get_splits()

type_tac = "ruff"
anomaly_rate = 0.1


emb_model = 'glove_300d.kv'
type_emb = 'glove'

emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)

inlier_topic = "crude"
# list_inlier_topic = ["earn", "acq", "crude", "trade", "money-fx", "interest", "ship"]


# for inlier_topic in list_inlier_topic:
# print(inlier_topic)
inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

data_test = tac.textual_anomaly_contamination(
                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)

inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)
data_test_emb = emb_encoder.forward(data_test)


X_train = np.array(inlier_dataset_train_emb[f'{type_emb}_embedding'])
X_test = np.array(data_test_emb[f'{type_emb}_embedding'])
y_true_test = np.array(data_test_emb['anomaly_class'])  
y_true_train = np.array(anomaly_dataset_train['anomaly_class'])  


In [ ]:
param_grid = {
    "kernel": ["rbf", "sigmoid"],
    "nu": [0.05, 0.1, 0.15],
    "gamma": [1e-3, 1e-2, 1e-1, 1]
}

best_params, results = ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid, n_splits=3)
print(best_params, end="\n")
# clf = OneClassSVM(**best_params)
clf, _, _ = ocsvm.One_Class_SVM(inlier_dataset_train_emb[f'{type_emb}_embedding'], best_params)
print(clf)
clf.fit(X_train)

scores_test = clf.decision_function(X_test)
# print(scores_test)
auc, ap, fpr95 = ev.evaluation(y_true_test, scores_test, verbose=True)
print("Final results with best params:", auc, ap, fpr95)

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

# === Param grid ===
param_grid = {
    "kernel": ["rbf", "sigmoid"],
    "nu": [0.05, 0.1, 0.15],
    "gamma": [1e-3, 1e-2, 1e-1, 1]
}

# === Datasets & inliers ===
list_dataset_name = ["20newsgroups", "reuters"]
list_list_inlier_topic = [
    ["computer", "recreation", "science", "miscellaneous", "politics", "religion"],
    ["earn", "acq", "crude", "trade", "money-fx", "interest", "ship"]
]

# === Embeddings ===
emb_model = 'glove_300d.kv'
type_emb = 'glove'

# dictionnaire final
ocsvm_params_dict = {}

# fonction cross-validation pour choisir les meilleurs params
def ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid):
    best_auc = -np.inf
    best_params = None
    for kernel in param_grid["kernel"]:
        for nu in param_grid["nu"]:
            for gamma in param_grid["gamma"]:
                # clf, _, _ = ocsvm.One_Class_SVM(inlier_dataset_train_emb[f'{type_emb}_embedding'], best_params)
                # clf.fit(X_train)

                ocsvm_kwargs = {
                    "nu": nu,
                    "kernel": kernel,
                    "gamma": gamma
                    }
                clf, _, _ = ocsvm.One_Class_SVM(X_train, ocsvm_kwargs)

                scores_test = clf.decision_function(X_test)
                auc = roc_auc_score(y_true_test, scores_test)
                if auc > best_auc:
                    best_auc = auc
                    best_params = {"kernel": kernel, "nu": nu, "gamma": gamma}
    return best_params, best_auc

# === Boucle sur datasets et inliers ===
for i, dataset_name in enumerate(list_dataset_name):
    list_inlier_topic = list_list_inlier_topic[i]
    ocsvm_params_dict[dataset_name] = {}

    dataset = Dataset.ADDataset(dataset_name, False,True)
    trainset, testset = dataset.get_splits()
    emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)

    for inlier_topic in tqdm(list_inlier_topic, desc=f"{dataset_name} inliers"):
        # Préparation train/test
        inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
            trainset, dataset_name, inlier_topic, type_tac="ruff", anomaly_rate=0.1, is_trainset=True
        )
        data_test = tac.textual_anomaly_contamination(
            testset, dataset_name, inlier_topic, type_tac="ruff", anomaly_rate=0.1, is_trainset=False
        )

        # Embeddings
        inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)
        data_test_emb = emb_encoder.forward(data_test)

        X_train = np.array(inlier_dataset_train_emb[f'{type_emb}_embedding'])
        X_test = np.array(data_test_emb[f'{type_emb}_embedding'])
        y_true_test = np.array(data_test_emb['anomaly_class'])

        # Cross-validation pour trouver les meilleurs params
        print(param_grid)
        best_params, best_auc = ocsvm_cross_validation(X_train, X_test, y_true_test, param_grid)

        ocsvm_params_dict[dataset_name][inlier_topic] = best_params
        print(f"{dataset_name} | {inlier_topic} -> {best_params} | AUC={best_auc:.4f}")

print("\n✅ Final OCSVM parameters dictionary:")
print(ocsvm_params_dict)


## OCSVM

In [ ]:
emb_model = "glove_300d.kv"
type_emb = "glove"

# emb_model = "fasttext_300d.kv"
# type_emb = "fasttext"

# emb_model = "distilbert-base-uncased"
# type_emb = "bert"

emb_encoder = embedding_encoder.EmbeddingEncoder(emb_model, type_emb)

In [ ]:
inlier_dataset_train_emb = emb_encoder.forward(inlier_dataset_train)

In [ ]:
inlier_dataset_train_emb['glove_embedding']

In [ ]:
ocsvm_kwargs = {
        "nu": 0.4,
        "kernel": 'rbf',
        "gamma": 'scale'
    }
clf, y_pred_train, scores_train = ocsvm.One_Class_SVM(inlier_dataset_train_emb['glove_embedding'],
                                                      ocsvm_kwargs
                                                         )

In [ ]:
ds = ConcatDataset([emb_encoder.forward(inlier_dataset_test), emb_encoder.forward(anomaly_dataset_test)])
ds

In [ ]:
inputs_test = [x['glove_embedding'] for x in ds]
labels_test = [y['anomaly_class'] for y in ds]

y_pred_test = clf.predict(inputs_test)           
scores_test = clf.decision_function(inputs_test)

auc, f1, precision, recall, fpr95 = ev.evaluation(labels_test, scores_test, y_pred_test, verbose=False)

print(clf, end="\n\n")

print(auc)
print(f1)
print(precision)
print(recall)
print(fpr95)


## CVDD

### loop

In [13]:
def initialize_context_vectors(net, train_loader):
    """
    Initialize the context vectors from an initial run of k-means++ on simple average sentence embeddings

    Returns
    -------
    centers : ndarray, [n_clusters, n_features]
    """
    # np.random.seed(42)
    # Get vector representations
    X = ()
    for data in train_loader:
        inputs, _, _, _ = data
        # text.shape = (sentence_length, batch_size)

        X_batch = net.pretrained_model(inputs)
        # X_batch.shape = (sentence_length, batch_size, embedding_size)

        # compute mean and normalize
        X_batch = torch.mean(X_batch, dim=0)
        X_batch = X_batch / torch.norm(X_batch, p=2, dim=1, keepdim=True).clamp(min=1e-08)
        X_batch[torch.isnan(X_batch)] = 0
        # X_batch.shape = (batch_size, embedding_size)

        X += (X_batch.cpu().data.numpy(),)

    X = np.concatenate(X)
    n_attention_heads = net.n_attention_heads

    kmeans = KMeans(n_clusters=n_attention_heads).fit(X)
    centers = kmeans.cluster_centers_ / np.linalg.norm(kmeans.cluster_centers_, ord=2, axis=1, keepdims=True)

    return centers

In [14]:
dataset_name = '20newsgroups'
full_dataset_ = False
preprocessing = True

dataset = Dataset.ADDataset(dataset_name, full_dataset_, preprocessing)
trainset, testset = dataset.get_splits()

inlier_topic = "computer"
type_tac = "ruff"
anomaly_rate = 0.1

inlier_dataset_train, anomaly_dataset_train = tac.textual_anomaly_contamination(
                                            trainset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=True)

data_test = tac.textual_anomaly_contamination(
                                            testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)

Repo card metadata block was not found. Setting CardData to empty.


In [15]:
corpus = inlier_dataset_train['text']
vocab = utils.build_vocab(corpus,min_freq=3)
seq_len = 500

cvdd_dataset_train = Dataset.ADdatasets.CVDDDatasetWrapper(inlier_dataset_train, embedding_type='glove', vocab=vocab, seq_len=seq_len)
cvdd_dataset_test = Dataset.ADdatasets.CVDDDatasetWrapper(data_test, embedding_type='glove', vocab=vocab, seq_len=seq_len)

pretrained_model = embedding_layer.EmbeddingFactory.create('glove',
                        glove_path='./Modelisation/Baselines/CVDD/embedding_models/glove.6B.300d.txt',
                        vocab=vocab,
                        embedding_dim=300,
                        trainable=True)

In [17]:
dl_train = DataLoader(cvdd_dataset_train, batch_size=64, shuffle=True)
dl_test = DataLoader(cvdd_dataset_test, batch_size=64,shuffle=True)

attention_size = 150 
n_attention_heads = 3
model = cvdd_Net.CVDDNet(pretrained_model, attention_size, n_attention_heads)

In [18]:
alpha_scheduler = 'logarithmic'
n_epochs = 100

# alpha annealing strategy
alpha_milestones = np.arange(1, 6) * int(n_epochs / 5)  # 5 equidistant milestones over n_epochs
if alpha_scheduler == 'soft':
    alphas = [0.0] * 5
if alpha_scheduler == 'linear':
    alphas = np.linspace(.2, 1, 5)
if alpha_scheduler == 'logarithmic':
    alphas = np.logspace(-4, 0, 5)
if alpha_scheduler == 'hard':
    alphas = [100.0] * 4

In [19]:
lr = 0.01
weight_decay = 1e-6
lr_milestones = [40, 60]
lambda_p = 1.0

train_dists = None
train_att_matrix = None
train_top_words = None
c = None


In [20]:
model.c.data = torch.from_numpy(
                initialize_context_vectors(model, dl_train)[np.newaxis, :])

parameters = filter(lambda p: p.requires_grad, model.parameters())
optimizer = optim.Adam(parameters, lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=lr_milestones, gamma=0.1)

In [21]:
model = model.to(device)
model.train()
alpha_i = 0

for epoch in range(n_epochs):
    
    # scheduler.step()

    if epoch in lr_milestones:
        print(f"LR scheduler: new learning rate is %g" % float(scheduler.get_last_lr()[0]))

    if epoch in alpha_milestones:
        model.alpha = float(alphas[alpha_i])
        print('  Temperature alpha scheduler: new alpha is %g' % model.alpha)
        alpha_i += 1

    epoch_loss = 0.0
    n_batches = 0
    att_matrix = np.zeros((n_attention_heads, n_attention_heads))
    dists_per_head = ()
    epoch_start_time = time.time()

    for inputs, labels, texts, idx in dl_train:

        inputs = inputs.transpose(0, 1).to(device)
        
        optimizer.zero_grad() 

        cosine_dists, context_weights, A = model(inputs)
        scores = context_weights * cosine_dists

        I = torch.eye(n_attention_heads).to(device)
        CCT = model.c @ model.c.transpose(1, 2)
        P = torch.mean((CCT.squeeze() - I) ** 2)


        loss_P = lambda_p * P
        loss_emp = torch.mean(torch.sum(scores, dim=1))
        loss = loss_emp + loss_P


        dists_per_head += (cosine_dists.cpu().data.numpy(),)


        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)  # clip gradient norms in [-0.5, 0.5]
        optimizer.step()

        AAT = A @ A.transpose(1, 2)
        att_matrix += torch.mean(AAT, 0).cpu().data.numpy()

        epoch_loss += loss.item()
        n_batches += 1


    epoch_train_time = time.time() - epoch_start_time
    print(f'| Epoch: {epoch + 1:03}/{n_epochs:03} | Train Time: {epoch_train_time:.3f}s '
    f'| Train Loss: {epoch_loss / n_batches:.6f} |')

    scheduler.step()


    train_dists = np.concatenate(dists_per_head)
    train_att_matrix = att_matrix / n_batches
    train_att_matrix = train_att_matrix.tolist()


c = np.squeeze(model.c.data.cpu().numpy())
c = c.tolist()


| Epoch: 001/100 | Train Time: 1.042s | Train Loss: 0.075680 |
| Epoch: 002/100 | Train Time: 0.794s | Train Loss: 0.013706 |
| Epoch: 003/100 | Train Time: 0.793s | Train Loss: 0.009070 |
| Epoch: 004/100 | Train Time: 0.795s | Train Loss: 0.007781 |
| Epoch: 005/100 | Train Time: 0.797s | Train Loss: 0.007110 |
| Epoch: 006/100 | Train Time: 0.795s | Train Loss: 0.006815 |
| Epoch: 007/100 | Train Time: 0.794s | Train Loss: 0.006596 |
| Epoch: 008/100 | Train Time: 0.793s | Train Loss: 0.006441 |
| Epoch: 009/100 | Train Time: 0.796s | Train Loss: 0.006365 |
| Epoch: 010/100 | Train Time: 0.791s | Train Loss: 0.006343 |
| Epoch: 011/100 | Train Time: 0.797s | Train Loss: 0.006281 |
| Epoch: 012/100 | Train Time: 0.791s | Train Loss: 0.006375 |
| Epoch: 013/100 | Train Time: 0.794s | Train Loss: 0.006341 |
| Epoch: 014/100 | Train Time: 0.795s | Train Loss: 0.006301 |
| Epoch: 015/100 | Train Time: 0.792s | Train Loss: 0.006298 |
| Epoch: 016/100 | Train Time: 0.788s | Train Loss: 0.0

In [22]:
c = np.squeeze(model.c.data.cpu().numpy())
c = c.tolist()

In [23]:
test_dists = None
test_att_matrix = None
test_top_words = None
test_auc = 0.0
test_scores = None
test_att_weights = None

ad_score = 'context_dist_mean'


In [24]:
auc_s = []
for _ in range(10):
    
    data_test = tac.textual_anomaly_contamination(
                                                testset, dataset_name, inlier_topic, type_tac, anomaly_rate, is_trainset=False)
    # print(np.unique(data_test['anomaly_class'], return_counts=True))
    # print(data_test.filter(lambda x: x['anomaly_class'] == 1)['text'] )
    # print("-------------------")
    cvdd_dataset_test = Dataset.ADdatasets.CVDDDatasetWrapper(data_test, embedding_type='glove', vocab=vocab, seq_len=seq_len)
    dl_test = DataLoader(cvdd_dataset_test, batch_size=64, shuffle=True)


    model.eval()


    n_attention_heads = model.n_attention_heads
    epoch_loss = 0.0
    n_batches = 0
    att_matrix = np.zeros((n_attention_heads, n_attention_heads))
    dists_per_head = ()
    idx_label_score_head = []
    att_weights = []
    att_weights_ = []
    start_time = time.time()
    model.eval()

    with torch.no_grad():
        for inputs, labels, texts, idx in dl_test:
            inputs = inputs.transpose(0, 1).to(device)
            cosine_dists, context_weights, A = model(inputs)
            # print(cosine_dists)
            # print(np.unique(labels,return_counts=True)[1])
            scores = context_weights * cosine_dists
            _, best_att_head = torch.min(scores, dim=1)
            # print(best_att_head == torch.min(cosine_dists, dim=1)[1])
            # _, best_att_head = torch.min(cosine_dists, dim=1)
            # print(best_att_head)
            # break
            I = torch.eye(n_attention_heads).to(device)
            CCT = model.c @ model.c.transpose(1, 2)
            P = torch.mean((CCT.squeeze() - I) ** 2)

            loss_P = lambda_p * P
            loss_emp = torch.mean(torch.sum(scores, dim=1))
            loss = loss_emp + loss_P

            dists_per_head += (cosine_dists.cpu().data.numpy(),)
            ad_scores = torch.mean(cosine_dists, dim=1)

            idx_label_score_head += list(zip(idx,
                                        labels.cpu().data.numpy().tolist(),
                                        ad_scores.cpu().data.numpy().tolist(),
                                        best_att_head.cpu().data.numpy().tolist()))

            att_weights_ += A[best_att_head][:][range(len(idx))].cpu().data.numpy().tolist()
            # att_weights += A[range(len(idx)), best_att_head].cpu().data.numpy().tolist()

            # batch_size = A.size(0)
            # batch_idx = torch.arange(batch_size, device=A.device)
            # # Select for each sample i the attention vector of the chosen head best_att_head[i]
            # selected_att = A[batch_idx, best_att_head, :]   # shape: (batch, seq_len)
            # att_weights += selected_att.cpu().numpy().tolist()

            # print(att_weights == att_weights_)

            AAT = A @ A.transpose(1, 2)
            att_matrix += torch.mean(AAT, 0).cpu().data.numpy()

            epoch_loss += loss.item()
            n_batches += 1



    # raise Exception("DAYEN !!!")
    test_dists = np.concatenate(dists_per_head)
    test_att_matrix = att_matrix / n_batches
    test_att_matrix = test_att_matrix.tolist()

    test_scores = idx_label_score_head
    test_att_weights = att_weights

    # Compute AUC
    _, labels, scores, _ = zip(*idx_label_score_head)
    labels = np.array(labels)
    scores = np.array(scores)
    # print(scores)
    # print(labels)

    if np.sum(labels) > 0:
        best_context = None
        if ad_score == 'context_dist_mean':
            test_auc = roc_auc_score(labels, scores)
        if ad_score == 'context_best':
            test_auc = 0.0
            for context in range(n_attention_heads):
                auc_candidate = roc_auc_score(labels, test_dists[:, context])
                if auc_candidate > test_auc:
                    test_auc = auc_candidate
                    best_context = context
                else:
                    pass
    else:
        best_context = None
        test_auc = 0.0


    # Log results
    print('Test Loss: {:.6f}'.format(epoch_loss / n_batches))
    print('Test AUC: {:.2f}%'.format(100. * test_auc))
    print(f'Test Best Context: {best_context}')
    print('Finished testing.\n')
    # print("-------------------------------")
    auc_s.append(test_auc)
    
auc_s = np.array(auc_s)

Test Loss: 0.009362
Test AUC: 53.57%
Test Best Context: None
Finished testing.

Test Loss: 0.009448
Test AUC: 53.91%
Test Best Context: None
Finished testing.

Test Loss: 0.008793
Test AUC: 52.55%
Test Best Context: None
Finished testing.

Test Loss: 0.009405
Test AUC: 51.29%
Test Best Context: None
Finished testing.

Test Loss: 0.009073
Test AUC: 53.42%
Test Best Context: None
Finished testing.

Test Loss: 0.008892
Test AUC: 50.47%
Test Best Context: None
Finished testing.

Test Loss: 0.009619
Test AUC: 52.49%
Test Best Context: None
Finished testing.

Test Loss: 0.009534
Test AUC: 53.39%
Test Best Context: None
Finished testing.

Test Loss: 0.009311
Test AUC: 51.23%
Test Best Context: None
Finished testing.

Test Loss: 0.009332
Test AUC: 53.09%
Test Best Context: None
Finished testing.



In [25]:
auc_s.mean(), auc_s.std()

(0.5254039624264852, 0.0110883182063963)

## Save

In [ ]:
import numpy as np
from sklearn.svm import OneClassSVM as SklearnOCSVM
from pyod.models.ocsvm import OCSVM as PyODOCSVM
from sklearn.metrics import roc_auc_score

# --- Création d'un dataset toy ---
np.random.seed(42)
X_train = 0.3 * np.random.randn(100, 2)           # données normales
X_test = np.r_[X_train, 2 + 0.3 * np.random.randn(20, 2)]  # ajouter anomalies
y_test = np.array([0]*100 + [1]*20)               # 0 = normal, 1 = anomalie

# --- Hyperparamètres communs ---
kernel = 'rbf'
gamma = 0.1
nu = 0.1  # proportion d'anomalies attendues

# --- Sklearn OCSVM ---
sk_ocsvm = SklearnOCSVM(kernel=kernel, gamma=gamma, nu=nu)
sk_ocsvm.fit(X_train)
sk_scores = -sk_ocsvm.decision_function(X_test)  # négatif car dans sklearn, + = normal, - = anomalie
sk_auc = roc_auc_score(y_test, sk_scores)

# --- PyOD OCSVM ---
pyod_ocsvm = PyODOCSVM(kernel=kernel, gamma=gamma, nu=nu)
pyod_ocsvm.fit(X_train)
pyod_scores = pyod_ocsvm.decision_function(X_test)  # déjà + = anomalie
pyod_auc = roc_auc_score(y_test, pyod_scores)

# --- Résultats ---
print(f"ROC AUC Sklearn OCSVM: {sk_auc:.4f}")
print(f"ROC AUC PyOD OCSVM:    {pyod_auc:.4f}")
